In [9]:
# For every (region, weather variable) pair: which stations combine.py actually picked,
# dist_snow_station_km / dist_precip_station_km columns in combined.csv, but for ALL variables.
import pandas as pd
from assign_regions import load_regions, nearest_stations
from combine import daily_station_features

df_stations = pd.read_csv("data/station_data.csv").drop(columns=["substation", "tb100"], errors="ignore")
df_metadata = pd.read_csv("data/stations_metadata.csv")[["id", "Stationsname", "Länge [°E]", "Breite [°N]", "Höhe [m]", "Bundesland"]]
df_metadata.columns = ["station", "name", "lon", "lat", "elevation_m", "state"]
df = df_stations.merge(df_metadata, on="station", how="left", validate="many_to_one")
RENAME = {"time": "timestamp", "tl": "air_temp_c", "rr": "precip_mm", "rrm": "precip_minutes",
          "sh": "snow_depth_cm", "ff": "wind_speed_ms", "ffx": "wind_gust_ms", "dd": "wind_dir_deg",
          "rf": "rel_humidity_pct", "p": "pressure_hpa", "so_h": "sunshine_h", "tb10": "soil_temp_10cm_c"}
df = df.rename(columns=RENAME)

regions = load_regions("data/AT-07_micro-regions.geojson.json")
stations = df.drop_duplicates("station")[["station", "name", "lon", "lat", "elevation_m"]]
wide = daily_station_features(df)

In [10]:
# same selection rule as combine.py: region_weather() -- k=3 nearest stations with >=30% day-coverage
K, MIN_COVERAGE = 3, 0.3
FEATURES = [f for f in wide if f not in ("wind_u", "wind_v")]  # those two are internal (used to build wind_dir_deg)

dist = nearest_stations(stations, regions, k=len(stations))

rows = []
for region, dd in dist.groupby("region"):
    dkm = dd.set_index("station").dist_km
    for feat in FEATURES:
        X = wide[feat]
        cand = [s for s in X.columns if X[s].notna().mean() >= MIN_COVERAGE]
        if not cand:
            rows.append({"region": region, "feature": feat, "n_candidates": 0, "max_dist_km": None, "stations": None})
            continue
        sel = dkm.loc[cand].nsmallest(K)
        rows.append({"region": region, "feature": feat, "n_candidates": len(cand),
                      "max_dist_km": round(float(sel.max()), 1), "stations": ", ".join(str(s) for s in sel.index)})

quality = pd.DataFrame(rows)
quality.head(10)

,region,feature,n_candidates,max_dist_km,stations
0,AT-07-01,air_temp_mean,12,28.6,"14308, 14305, 17005"
1,AT-07-01,air_temp_min,12,28.6,"14308, 14305, 17005"
2,AT-07-01,air_temp_max,12,28.6,"14308, 14305, 17005"
3,AT-07-01,wind_speed_mean,12,28.6,"14308, 14305, 17005"
4,AT-07-01,wind_gust_max,12,28.6,"14308, 14305, 17005"
5,AT-07-01,rel_humidity_mean,12,28.6,"14308, 14305, 17005"
6,AT-07-01,pressure_mean,10,30.9,"14305, 17005, 28"
7,AT-07-01,sunshine_sum,12,28.6,"14308, 14305, 17005"
8,AT-07-01,snow_depth_cm,5,62.9,"14305, 184, 66"
9,AT-07-01,precip_sum_mm,9,30.9,"14305, 17005, 28"


In [11]:
# worst-case ("weakest link") station distance per region, across all variables -- sorted, worst first
worst_per_region = quality.groupby("region").max_dist_km.max().sort_values(ascending=False)
worst_per_region.head(15)

region
AT-07-29-03    137.4
AT-07-28       119.2
AT-07-18       116.0
AT-07-26-02    115.3
AT-07-26       114.8
AT-07-26-01    114.8
AT-07-06       114.5
AT-07-29-02    110.3
AT-07-29-01    103.1
AT-07-29       103.1
AT-07-17-02    100.3
AT-07-27        89.3
AT-07-25        84.8
AT-07-24        84.7
AT-07-05        80.6
Name: max_dist_km, dtype: float64

In [12]:
# which (region, feature) pairs have NO usable station at all (0 candidates)?
none_available = quality[quality.n_candidates == 0]
print(f"{len(none_available)} region-feature pairs with no usable station")
none_available[["region", "feature"]]

0 region-feature pairs with no usable station


,region,feature
